# 01 - Data Cleaning
## Global Superstore Dataset (2011-2014)

**الهدف من النوتبوك ده:** تحويل الداتا الخام لداتا نظيفة وجاهزة للتحليل، مع توثيق سبب كل خطوة.

## 1. Load Raw Data

In [ ]:
import pandas as pd
import numpy as np
import sys
sys.path.insert(0, '../src')

raw_data = pd.read_csv('../data/raw/superstore_dataset2011-2015.csv', encoding='latin1')
print(raw_data.shape)
raw_data.head()

## 2. Initial Inspection

فحص الأعمدة، القيم المفقودة، والتكرار قبل أي تعديل.

In [ ]:
print(raw_data.dtypes)
print()
print('Missing values:')
print(raw_data.isnull().sum()[raw_data.isnull().sum() > 0])
print()
print('Duplicates:', raw_data.duplicated().sum())

## 3. Parsing Mixed-Format Dates

**اكتشاف مهم:** عمود التاريخ فيه فورمات مختلط (`/` و `-`). اتأكدنا من الترتيب الصحيح (DD/MM/YYYY)
باستخدام قاعدة منطقية: تاريخ الشحن لازم يكون بعد أو يساوي تاريخ الطلب أبدًا قبله.

In [ ]:
cleaned_data = raw_data.copy()

cleaned_data['Order Date'] = pd.to_datetime(
    cleaned_data['Order Date'], dayfirst=True, format='mixed'
)
cleaned_data['Ship Date'] = pd.to_datetime(
    cleaned_data['Ship Date'], dayfirst=True, format='mixed'
)

# Validation rule: Ship Date must never precede Order Date
invalid_dates = (cleaned_data['Ship Date'] < cleaned_data['Order Date']).sum()
assert invalid_dates == 0, f'{invalid_dates} rows have invalid date order!'
print('Date parsing validated successfully. Invalid rows:', invalid_dates)

## 4. Feature Engineering: Processing Time

عمود مشتق جديد: عدد الأيام بين تاريخ الطلب وتاريخ الشحن.

In [ ]:
cleaned_data['Processing Time'] = (
    cleaned_data['Ship Date'] - cleaned_data['Order Date']
).dt.days

print(cleaned_data[['Order Date', 'Ship Date', 'Processing Time']].head())
print()
print(cleaned_data.groupby('Ship Mode')['Processing Time'].mean().sort_values())

## 5. Dropping Non-Analytical Columns

- `Row ID`: معرّف تسلسلي بس، مالوش قيمة تحليلية
- `Postal Code`: 80% من قيمه فاضية، ومش هيفيد التحليل (عندنا City/State/Region/Market بديل)

In [ ]:
cleaned_data = cleaned_data.drop(columns=['Row ID', 'Postal Code'])
print('Shape after cleaning:', cleaned_data.shape)

## 6. Final Quality Check

In [ ]:
print('Duplicates:', cleaned_data.duplicated().sum())
print('Nulls remaining:')
print(cleaned_data.isnull().sum()[cleaned_data.isnull().sum() > 0])
print()
print('Categorical values sanity check:')
for col in ['Market','Region','Category','Sub-Category','Segment','Ship Mode','Order Priority']:
    print(col, '->', sorted(cleaned_data[col].unique()))

## 7. Save Cleaned Data

In [ ]:
cleaned_data.to_csv('../data/processed/cleaned_superstore.csv', index=False)
print('Saved to data/processed/cleaned_superstore.csv')